# Regressão logística: o caso da inadimplência

Capítulo II.3 do livro vivo [Ciência de Dados e Aprendizado de Máquina](https://machinelearning.ghdaru.com.br/ii-3-regressao-logistica.html).

Um banco, 4 000 clientes, uma pergunta por fatura: este aqui vai pagar? Você vai responder primeiro do jeito errado, com uma reta, e ver o número que ela devolve. Depois do jeito certo, e descobrir que o jeito certo também engana se você ler a acurácia sozinha.

Leia o capítulo antes. Ele explica o porquê; aqui você mexe nos números.

Roda sem instalar nada: biblioteca padrão do Python e o código do próprio livro. Nem NumPy, que só entra na etapa 09.

No fim, responda o questionário no Forms que o professor vai compartilhar.

In [ ]:
# --- roda igual na sua máquina e no Colab ------------------------------
# Na sua máquina: o notebook acha o repositório subindo de pasta.
# No Colab: não há repositório, então os arquivos necessários são baixados.
import pathlib, sys, urllib.request

RAW = "https://raw.githubusercontent.com/GHDaru/machinelearning/main/"
PRECISA = ['ml-zero/etapa-05/linear.py', 'ml-zero/etapa-00/dados.py']

raiz = pathlib.Path.cwd()
for _ in range(5):
    if (raiz / "ml-zero").is_dir():
        break
    raiz = raiz.parent
else:
    raiz = pathlib.Path.cwd()

for rel in PRECISA:
    destino = raiz / rel
    if not destino.exists():
        destino.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(RAW + rel, destino)
        print("baixado:", rel)

sys.path.insert(0, str(raiz / "ml-zero/etapa-05"))
sys.path.insert(0, str(raiz / "ml-zero/etapa-00"))
print("pronto.")

## 1. O dado

Três atributos por cliente: `renda_mensal`, `meses_de_relacionamento`, `uso_do_limite`. Rótulo 1 é inadimplência.

O conjunto é sintético, gerado em `etapa-00/dados.py` com seed fixa. Isso tem um preço (dado sintético não tem as sujeiras do dado real) e uma vantagem que vamos cobrar na seção 4: nós sabemos a resposta certa.

In [ ]:
from dados import gerar, dividir

dados = gerar(n=4000)
treino, validacao, teste = dividir(dados)

print(f"{len(dados)} clientes · prevalência de inadimplência {dados.prevalencia():.3f}")
print(f"treino {len(treino)} · validação {len(validacao)} · teste {len(teste)}")
print("\nprimeiro cliente:", dict(zip(dados.nomes, dados.X[0])), "-> rótulo", dados.y[0])

19% de inadimplentes. Guarde esse número; ele volta duas vezes.

## 2. A tentação: uma reta

Pagou é 0, não pagou é 1. É uma coluna numérica como qualquer outra. Por que não ajustar uma regressão linear e ler a saída como probabilidade?

In [ ]:
from linear import RegressaoLinear

Xt = [[c[0], c[1], c[2]] for c in treino.X]
yt = [float(v) for v in treino.y]

linha = RegressaoLinear(solucao_fechada=True, padronizar=True).fit(Xt, yt)
pred_linha = linha.predict(Xt)

fora_do_intervalo = sum(1 for p in pred_linha if p < 0 or p > 1)
print(f"no treino: {fora_do_intervalo} de {len(pred_linha)} previsões já saem de [0,1] "
      f"({fora_do_intervalo / len(pred_linha):.1%})")

# um cliente só um pouco mais extremo que o conjunto, nada absurdo:
ultra_seguro = [50_000.0, 240.0, 0.0]   # renda alta, 20 anos de relacionamento, limite sem uso
print(f"cliente muito seguro {ultra_seguro} -> "
      f"'probabilidade' prevista de {linha.predict([ultra_seguro])[0]:+.3f}")

Quinze clientes do treino já saem de [0,1]. E o cliente da última linha, que não tem nada de absurdo (renda alta, vinte anos de casa, limite sem uso), recebe **−0,757**. Menos setenta e cinco por cento de probabilidade de não pagar.

A reta não sabe que está prevendo uma probabilidade. Ela só sabe seguir em frente. O que falta é uma função que trave em 0 e em 1 por mais que o cliente melhore ou piore. Essa função é a sigmoide.

## 3. A logística

`RegressaoLogistica` é a classe do capítulo: a mesma reta, passada pela sigmoide, treinada por gradiente sobre a entropia cruzada. Gradiente porque não há outro jeito. Anule a derivada e os pesos ficam presos dentro da sigmoide, sem álgebra que os tire de lá.

In [ ]:
from linear import RegressaoLogistica

Xv = [[c[0], c[1], c[2]] for c in validacao.X]
yv = [float(v) for v in validacao.y]

modelo = RegressaoLogistica(taxa=0.5, epocas=400, validacao=(Xv, yv),
                             paciencia=20, seed=0).fit(Xt, yt)

print(f"treinou {modelo.historico.epocas} épocas"
      + (" (parou cedo)" if modelo.historico.parou_cedo else ""))
print(f"perda final de treino: {modelo.historico.perdas[-1]:.4f}")

58 épocas e parou sozinha, porque a perda na validação deixou de cair. Agora o teste.

In [ ]:
Xte = [[c[0], c[1], c[2]] for c in teste.X]
yte = teste.y

pred = modelo.predict(Xte)   # limiar padrão: 0,5
acertos = sum(1 for p, y in zip(pred, yte) if p == y)
acuracia = acertos / len(yte)
baseline = 1 - teste.prevalencia()  # sempre prever a classe majoritária (0)

vp = sum(1 for p, y in zip(pred, yte) if p == 1 and y == 1)
fn = sum(1 for p, y in zip(pred, yte) if p == 0 and y == 1)
revocacao = vp / (vp + fn) if (vp + fn) else float("nan")

print(f"acurácia (limiar 0,5):   {acuracia:.4f}   (linha de base: {baseline:.4f})")
print(f"revocação da classe 1:   {revocacao:.4f}   (de {vp + fn} inadimplentes, achou {vp})")

**0,8092 contra 0,8092.** O modelo empatou com um "modelo" que diz não para todo mundo. A segunda linha explica o empate: 153 inadimplentes no teste, zero encontrados.

Não é que ele não aprendeu nada. É que quase nenhum cliente passa de 0,5 de probabilidade quando só 19% da base é inadimplente (o número que você guardou). O limiar 0,5 é um padrão da biblioteca, não uma decisão sua. Decida:

In [ ]:
limiar = teste.prevalencia()
pred2 = [1 if p > limiar else 0 for p in modelo.predict_proba(Xte)]
acuracia2 = sum(1 for p, y in zip(pred2, yte) if p == y) / len(yte)
vp2 = sum(1 for p, y in zip(pred2, yte) if p == 1 and y == 1)
fn2 = sum(1 for p, y in zip(pred2, yte) if p == 0 and y == 1)
revocacao2 = vp2 / (vp2 + fn2)

print(f"limiar = prevalência ({limiar:.3f})")
print(f"acurácia:   {acuracia2:.4f}")
print(f"revocação:  {revocacao2:.4f}   (de {vp2 + fn2} inadimplentes, achou {vp2})")

Acurácia foi a 0,574; revocação, a 0,608. Dos 153, agora acha 93, e incomoda um bom tanto de bons pagadores no caminho. Se a troca vale, depende do que custa cada erro, e isso é o capítulo II.1.

> Acurácia sozinha teria enterrado um modelo que funcionava.

## 4. O coeficiente é sobre chance, não sobre probabilidade

Os pesos saem na escala padronizada: um desvio-padrão do atributo. `razao_de_chances()` devolve `exp(w)`, que é por quanto a chance de inadimplência é multiplicada. Chance, não probabilidade.

In [ ]:
for nome, w, odds in zip(dados.nomes, modelo.pesos, modelo.razao_de_chances()):
    sinal = "aumenta" if odds > 1 else "reduz"
    print(f"{nome:24s} w={w:+.3f}   razão de chances={odds:.3f}  "
          f"({sinal} a chance em {abs(odds - 1):.0%} por desvio-padrão)")

Um desvio-padrão a mais de uso do limite multiplica a chance por 1,38; um de renda, por 0,78. Agora abra `etapa-00/dados.py` e leia a linha que define `z`: uso para cima, log da renda e meses para baixo. Os três sinais batem. As magnitudes não precisam bater: o gerador usa `log(renda)` e o modelo só viu a renda bruta.

Esse é o luxo do dado sintético. Com dado real você teria os mesmos três números e nenhum gabarito para comparar.

## 5. O mesmo coeficiente, dois clientes

O exercício e9 do capítulo, em código. Dois clientes recebem o mesmo aumento no uso do limite, trinta pontos. Um partia de risco baixo, o outro de risco alto.

In [ ]:
def cliente(renda, meses, uso):
    return [renda, meses, uso]

baixo_risco = cliente(renda=7000, meses=60, uso=0.10)
alto_risco  = cliente(renda=2500, meses=4,  uso=0.60)

for nome, c in [("baixo risco", baixo_risco), ("alto risco", alto_risco)]:
    antes = modelo.predict_proba([c])[0]
    depois_X = [c[0], c[1], min(1.0, c[2] + 0.30)]   # +30 pontos de uso do limite
    depois = modelo.predict_proba([depois_X])[0]
    print(f"{nome:12s}  p(inadimplência) {antes:.3f} -> {depois:.3f}   "
          f"(deslocamento de {depois - antes:+.3f})")

+0,050 para um, +0,117 para o outro. O coeficiente foi o mesmo; o efeito na probabilidade, mais que o dobro. É a sigmoide: íngreme no meio, quase plana perto de zero. Quem escreve num relatório "cada ponto de uso do limite aumenta o risco em X%" está dizendo algo que só vale para um cliente específico, e não disse qual.

Troque os dois clientes e veja o padrão se repetir.

## O que levar

- A reta entregou −0,757 como probabilidade e não piscou. A sigmoide existe para isso não acontecer.
- O modelo logístico empatou com a linha de base na acurácia e achou zero inadimplentes. Os dois números juntos contam a história; qualquer um sozinho mente.
- O coeficiente multiplica a chance. O efeito na probabilidade depende de onde o cliente estava, e aqui foi de 0,050 a 0,117 com o mesmo peso.
- Com dado sintético, os sinais dos coeficientes puderam ser conferidos contra o gerador. Com dado real não haveria contra o que conferir.

Volte ao capítulo [II.3 — Regressão Logística](https://machinelearning.ghdaru.com.br/ii-3-regressao-logistica.html) e responda o questionário que o professor vai compartilhar.